05_robustness_testing.ipynb - Robustness & Adversarial Testing


In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [ ]:
# ── 1. Load data & model ─────────────────────────────────────────
df = pd.read_csv('../data/processed/train_feat.csv')
X  = df.drop(columns=['Survived'])
y  = df['Survived']

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = joblib.load('../models/randomforest.joblib')
print(f"Test set shape: {X_test.shape}")
print(f"Column names: {X_test.columns.tolist()}")


测试集维度: (179, 8)
列名: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']


In [ ]:
# ── 2. Baseline performance ────────────────────────────────────────────
def evaluate(model, X, y, label='baseline'):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    acc    = accuracy_score(y, y_pred)
    auc    = roc_auc_score(y, y_prob)
    print(f"[{label:35s}]  acc={acc:.4f}  auc={auc:.4f}")
    return {'label': label, 'acc': acc, 'auc': auc}

results = []
results.append(evaluate(model, X_test, y_test, 'baseline'))


[baseline                           ]  acc=0.8156  auc=0.8339


In [ ]:
# ── 3. Perturbation functions ───────────────────────────────────

def add_gaussian_noise(X, cols, noise_std=0.5):
    """Add Gaussian noise to numeric columns"""
    X_p = X.copy()
    for col in cols:
        if col in X_p.columns:
            X_p[col] += np.random.normal(0, noise_std, size=len(X_p))
    return X_p

def random_mislabel(X, cols, flip_ratio=0.2):
    """Randomly flip binary class labels"""
    X_p = X.copy()
    for col in cols:
        if col in X_p.columns:
            mask = np.random.rand(len(X_p)) < flip_ratio
            X_p.loc[mask, col] = 1 - X_p.loc[mask, col]
    return X_p

def extreme_values(X, col_val_map):
    """Replace specified columns with extreme values"""
    X_p = X.copy()
    for col, val in col_val_map.items():
        if col in X_p.columns:
            X_p[col] = val
    return X_p


In [ ]:
# ── 4. Execute perturbation tests ────────────────────────────────
np.random.seed(42)

# 4-1 Age with noise (std=0.5)
X_p = add_gaussian_noise(X_test, ['Age'], noise_std=0.5)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_age_std0.5'))

# 4-2 Fare with noise (std=0.5)
X_p = add_gaussian_noise(X_test, ['Fare'], noise_std=0.5)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_fare_std0.5'))

# 4-3 Age + Fare with noise simultaneously
X_p = add_gaussian_noise(X_test, ['Age', 'Fare'], noise_std=0.5)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_age+fare_std0.5'))

# 4-4 Strong noise (std=2.0)
X_p = add_gaussian_noise(X_test, ['Age', 'Fare'], noise_std=2.0)
results.append(evaluate(model, X_p, y_test, 'gaussian_noise_age+fare_std2.0'))

# 4-5 Sex_male random flip 20%
X_p = random_mislabel(X_test, ['Sex_male'], flip_ratio=0.2)
results.append(evaluate(model, X_p, y_test, 'mislabel_sex_20pct'))

# 4-6 Embarked random flip 20%
X_p = random_mislabel(X_test, ['Embarked_Q', 'Embarked_S'], flip_ratio=0.2)
results.append(evaluate(model, X_p, y_test, 'mislabel_embarked_20pct'))

# 4-7 Extreme values: Age=0
X_p = extreme_values(X_test, {'Age': 0})
results.append(evaluate(model, X_p, y_test, 'extreme_age=0'))

# 4-8 Extreme values: Fare=0
X_p = extreme_values(X_test, {'Fare': 0})
results.append(evaluate(model, X_p, y_test, 'extreme_fare=0'))

# 4-9 Extreme values: Age=0 and Fare=0
X_p = extreme_values(X_test, {'Age': 0, 'Fare': 0})
results.append(evaluate(model, X_p, y_test, 'extreme_age=0_fare=0'))


[gaussian_noise_age_std0.5          ]  acc=0.7486  auc=0.8003
[gaussian_noise_fare_std0.5         ]  acc=0.7598  auc=0.8155
[gaussian_noise_age+fare_std0.5     ]  acc=0.7318  auc=0.7838
[gaussian_noise_age+fare_std2.0     ]  acc=0.7207  auc=0.8044
[mislabel_sex_20pct                 ]  acc=0.7318  auc=0.7507
[mislabel_embarked_20pct            ]  acc=0.8268  auc=0.8232
[extreme_age=0                      ]  acc=0.7486  auc=0.7686
[extreme_fare=0                     ]  acc=0.7989  auc=0.8747
[extreme_age=0_fare=0               ]  acc=0.7989  auc=0.7752


In [ ]:
# ── 5. Summarize results ────────────────────────────────────────────
df_results = pd.DataFrame(results)
print("\n── Summary ──")
print(df_results.to_string(index=False))

baseline_acc = df_results.loc[0, 'acc']
baseline_auc = df_results.loc[0, 'auc']
df_results['acc_drop'] = baseline_acc - df_results['acc']
df_results['auc_drop'] = baseline_auc - df_results['auc']



── 汇总 ──
                         label      acc      auc
                      baseline 0.815642 0.833860
     gaussian_noise_age_std0.5 0.748603 0.800264
    gaussian_noise_fare_std0.5 0.759777 0.815481
gaussian_noise_age+fare_std0.5 0.731844 0.783794
gaussian_noise_age+fare_std2.0 0.720670 0.804414
            mislabel_sex_20pct 0.731844 0.750725
       mislabel_embarked_20pct 0.826816 0.823188
                 extreme_age=0 0.748603 0.768577
                extreme_fare=0 0.798883 0.874704
          extreme_age=0_fare=0 0.798883 0.775165


In [ ]:
# ── 6. Assertion tests ────────────────────────────────────────────
print("\n── Assertion tests ──")

# Weak noise assertion <10%  
weak_noise = df_results[df_results['label'] == 'gaussian_noise_age+fare_std0.5'].iloc[0]
assert weak_noise['acc_drop'] < 0.15, \
    f"[FAIL] Weak noise acc drop too large: {weak_noise['acc_drop']:.4f}"
print(f"[PASS] Weak noise acc drop: {weak_noise['acc_drop']:.4f} < 0.15")

# Add strong noise assertion
strong_noise = df_results[df_results['label'] == 'gaussian_noise_age+fare_std2.0'].iloc[0]
assert strong_noise['acc_drop'] < 0.20, \
    f"[FAIL] Strong noise acc drop too large: {strong_noise['acc_drop']:.4f}"
print(f"[PASS] Strong noise acc drop: {strong_noise['acc_drop']:.4f} < 0.20")



── 断言测试 ──
[PASS] 弱噪声 acc 下降: 0.0838 < 0.15
[PASS] 强噪声 acc 下降: 0.0950 < 0.20


In [ ]:
# ── 7. Visualization & Save ───────────────────────────────────────
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels_short = [l.replace('gaussian_noise_', 'noise_')
                  .replace('mislabel_', 'mis_')
                  .replace('extreme_', 'ext_')
                for l in df_results['label']]

axes[0].barh(labels_short, df_results['acc'], color='steelblue')
axes[0].axvline(baseline_acc, color='red', linestyle='--', label='baseline')
axes[0].set_title('Accuracy under Perturbations')
axes[0].set_xlabel('Accuracy')
axes[0].legend()

axes[1].barh(labels_short, df_results['auc'], color='darkorange')
axes[1].axvline(baseline_auc, color='red', linestyle='--', label='baseline')
axes[1].set_title('AUC under Perturbations')
axes[1].set_xlabel('AUC')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/robustness_plot.png', dpi=150)
plt.show()
print("[PASS] Chart saved: results/robustness_plot.png")


[PASS] 图表已保存: results/robustness_plot.png


C:\Users\fan-z\AppData\Local\Temp\ipykernel_19388\2581042319.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ── 8. Write robustness.txt ─────────────────────────────────
with open('../results/robustness.txt', 'w', encoding='utf-8') as f:
    f.write("Robustness Test Results\n")
    f.write("=" * 60 + "\n")
    f.write(f"baseline_acc={baseline_acc:.4f}  baseline_auc={baseline_auc:.4f}\n\n")
    for _, row in df_results.iterrows():
        f.write(
            f"{row['label']:40s}  "
            f"acc={row['acc']:.4f}  "
            f"auc={row['auc']:.4f}  "
            f"acc_drop={row['acc_drop']:+.4f}  "
            f"auc_drop={row['auc_drop']:+.4f}\n"
        )

print("[PASS] Results written to: results/robustness.txt")


[PASS] 结果已写入: results/robustness.txt
